# E1 — HHL sobre una LS-SVM pequeña (métricas de recursos)

Replica la afirmación del informe: la SVM basada en HHL (Rebentrost et al. 2014) solo se ha
ejecutado en sistemas mínimos, es sensible al ruido y su costo en circuitos es alto.

**Qué hace este notebook:** construye el sistema lineal de una LS-SVM (Suykens),
`F · [b; α] = [0; y]`, implementa HHL a mano (QPE con reloj en complemento a dos, rotación
condicionada, QPE inversa, postselección), lo transpila a `["sx", "rz", "cx"]` y guarda una
fila de métricas por corrida en `results/hhl/<timestamp>/runs.csv`.

**Aviso:** el operador `e^{iFt}` se construye **clásicamente** (matriz exponencial o Trotter
sobre la matriz densa `F`). La profundidad del circuito simulado **no** es evidencia de
aceleración cuántica. Además, el bloque de rotaciones condicionadas es exponencial en `n_c`
(una RY multicontrolada por cada valor del reloj); es aceptable para `n_c ≤ 6`.

Todo es simulación (statevector / Aer). No se generan gráficos ni conclusiones.

Secciones: 1 Configuración · 2 Utilidades · 3 LS-SVM · 4 HHL · 5 Recursos · 6 Una corrida ·
7 Pruebas · 8 Barrido · 9 Vista previa.

## 1. Configuración

In [ ]:
import itertools, json, logging, platform, sys, time, csv
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.linalg as sla
from tqdm.auto import tqdm

from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit.library import PauliEvolutionGate, QFTGate, RYGate, StatePreparation, UnitaryGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter

# ----------------------------- Parámetros de ejecución -----------------------------
QUICK = True          # True: config rápida (1 corrida). False: barrido completo.
RESUME = False        # True: retoma OUTPUT_DIR saltando las corridas ya guardadas.
OUTPUT_DIR = None     # None -> results/hhl/<timestamp>. Con RESUME=True, indicar la carpeta a retomar.
NOISE = False         # True: además simula ruido depolarizante (matriz de densidad, solo N=3).
# -----------------------------------------------------------------------------------

CONFIG_FULL = dict(
    N=[3, 7],                 # N=15 (16x16) es opcional: agregar 15 a la lista
    n_c=[3, 4, 5, 6],
    gamma=[0.1, 1, 10],
    mode=["exact", "trotter"],
    seeds=[0, 1, 2],          # semilla de la submuestra
    kernel="linear",          # "linear" (Rebentrost) o "rbf"
    trotter_order=2,
    trotter_reps=1,           # pasos de Trotter para U^1; para U^(2^j) se usan reps * 2^j
    n_test=40,
    transpile_seed=0,
    noise=dict(p1=0.001, p2=0.01),
)
CONFIG_QUICK = {**CONFIG_FULL, "N": [3], "n_c": [4], "gamma": [1], "mode": ["exact"], "seeds": [0]}
CONFIG = CONFIG_QUICK if QUICK else CONFIG_FULL
CONFIG = {**CONFIG, "quick": QUICK, "noise_enabled": NOISE}

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("ROOT:", ROOT)
print(json.dumps(CONFIG, indent=2))

## 2. Utilidades (semillas, versiones, logging, checkpoint)

In [ ]:
import random

def set_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)

def package_versions():
    import importlib.metadata as md_
    names = ["numpy", "scipy", "pandas", "scikit-learn", "qiskit", "qiskit-aer", "qiskit-machine-learning"]
    return {n: md_.version(n) for n in names}

def setup_output(output_dir, resume):
    if output_dir is None:
        out = ROOT / "results" / "hhl" / datetime.now().strftime("%Y%m%d_%H%M%S")
    else:
        out = Path(output_dir)
        if not out.is_absolute():
            out = ROOT / out
    if out.exists() and not resume and any(out.iterdir()):
        raise FileExistsError(f"{out} ya existe; usar RESUME=True o otro OUTPUT_DIR")
    out.mkdir(parents=True, exist_ok=True)
    return out

def get_logger(out_dir):
    log = logging.getLogger("hhl")
    log.setLevel(logging.INFO)
    log.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    for h in (logging.StreamHandler(), logging.FileHandler(out_dir / "run.log", encoding="utf-8")):
        h.setFormatter(fmt)
        log.addHandler(h)
    return log

FIELDS = [
    "N", "dim_F", "n_system", "n_clock", "n_total_qubits", "mode", "gamma", "kernel", "seed",
    "kappa", "eigenvalues", "t", "train_idx",
    "depth_pre", "depth_post", "size_post", "cx_count", "gate_counts",
    "success_prob", "fidelity", "imag_residual",
    "acc_hhl", "acc_classical", "agreement",
    "fidelity_noisy", "success_prob_noisy",
    "wall_time_s",
]

def run_key(N, n_c, gamma, mode, seed, kernel):
    return f"{int(N)}|{int(n_c)}|{float(gamma)}|{mode}|{int(seed)}|{kernel}"

def load_done(csv_path):
    if not csv_path.exists():
        return set()
    df = pd.read_csv(csv_path)
    return {run_key(r.N, r.n_clock, r.gamma, r["mode"], r.seed, r.kernel) for _, r in df.iterrows()}

def append_row(csv_path, row):
    # checkpoint: cada corrida se guarda apenas termina
    new = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow({k: row.get(k) for k in FIELDS})

## 3. LS-SVM (sistema lineal y solución clásica)

```
F = [[0, 1ᵀ],            F · [b; α] = [0; y]
     [1, K + γ⁻¹ I]]
```
Si `N + 1` no es potencia de 2, `F` se rellena con un **bloque identidad** y el lado derecho con
ceros (los valores propios añadidos son 1 y no afectan a `[b; α]`). Datos: submuestra
estratificada de `breast_cancer`; estandarización y PCA (2 componentes) ajustadas solo con los
puntos de entrenamiento; `y ∈ {−1, +1}`.

In [ ]:
def make_kernel(kind, X_train):
    # kernel lineal (Rebentrost) o RBF con gamma = 'scale' de scikit-learn sobre el entrenamiento
    if kind == "linear":
        return lambda A, B: A @ B.T
    if kind == "rbf":
        g = 1.0 / (X_train.shape[1] * X_train.var())
        def k(A, B):
            d2 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
            return np.exp(-g * d2)
        return k
    raise ValueError(kind)

def load_subsample(N, seed, n_test):
    X, y = load_breast_cancer(return_X_y=True)
    y = 2 * y - 1                                   # {0,1} -> {-1,+1}
    idx = np.arange(len(y))
    for attempt in range(100):                      # reintenta hasta que el train tenga ambas clases
        tr, te = train_test_split(idx, train_size=N, test_size=n_test, stratify=y,
                                  random_state=seed + 1000 * attempt)
        if len(set(y[tr])) == 2:
            break
    else:
        raise RuntimeError("no se obtuvo un entrenamiento con ambas clases")
    scaler = StandardScaler().fit(X[tr])
    pca = PCA(n_components=2).fit(scaler.transform(X[tr]))
    f = lambda A: pca.transform(scaler.transform(A))
    return f(X[tr]), y[tr].astype(float), f(X[te]), y[te].astype(float), tr

def build_lssvm(X, y, gamma, kernel="linear"):
    # devuelve F (rellenada a potencia de 2) y el lado derecho rhs
    N = len(y)
    K = make_kernel(kernel, X)(X, X)
    F = np.zeros((N + 1, N + 1))
    F[0, 1:] = 1.0
    F[1:, 0] = 1.0
    F[1:, 1:] = K + np.eye(N) / gamma
    rhs = np.concatenate([[0.0], y])
    dim = 1 << int(np.ceil(np.log2(N + 1)))
    if dim > N + 1:                                 # relleno: bloque identidad y rhs con ceros
        Fp = np.eye(dim)
        Fp[: N + 1, : N + 1] = F
        F = Fp
        rhs = np.concatenate([rhs, np.zeros(dim - N - 1)])
    return F, rhs

def solve_classical(F, rhs):
    return np.linalg.solve(F, rhs)

def predict(x_vec, X_train, X_test, kernel="linear"):
    # sign(b + sum_i alpha_i k(x_i, x)); invariante a escala positiva del vector solución
    N = len(X_train)
    b, alpha = np.real(x_vec[0]), np.real(x_vec[1 : N + 1])
    s = b + make_kernel(kernel, X_train)(X_test, X_train) @ alpha
    return np.where(s >= 0, 1.0, -1.0)

## 4. Circuito HHL

Registros (little-endian de Qiskit): sistema `0..n_s-1`, reloj `n_s..n_s+n_c-1`, ancilla al final.

1. `StatePreparation` de `|b⟩` normalizado.
2. QPE sobre `U = e^{iFt}` con `n_c` qubits de reloj: `exact` (`UnitaryGate(expm(iFt·2^j))`
   controlada) o `trotter` (`PauliEvolutionGate` + Suzuki–Trotter, con `reps·2^j` pasos para `U^{2^j}`).
3. Rotación sobre el ancilla. El reloj se lee en **complemento a dos**: el valor `c ∈ [0, 2^{n_c})`
   representa `k = c` si `c < 2^{n_c-1}`, o `k = c − 2^{n_c}` si no, y
   `λ̃_k = 2π·k / (2^{n_c}·t)`. Para cada `k ≠ 0`, una RY multicontrolada con ángulo
   `2·arcsin(C/λ̃_k)` con `C = min|λ̃_k| = 2π/(2^{n_c}·t)`, es decir `C/λ̃_k = 1/k`.
   Para `k = 0` no hay rotación. **Es exponencial en `n_c`** (2^{n_c}−1 compuertas multicontroladas).
4. QPE inversa.
5. Postselección ancilla = 1 y reloj = 0; el vector se extrae de la `Statevector` (sin muestreo).

**Elección de `t`:** el mayor `|λ|` debe caer en el rango representable
`[−2^{n_c−1}, 2^{n_c−1}−1] / 2^{n_c}` de la fase `λt/2π`. Se usa
`t = 2π · (2^{n_c−1} − 1)/2^{n_c} / max|λ|`, de modo que `max|λ|` se asigna a `k = 2^{n_c−1} − 1`.
Los valores propios con `|λ|` tan pequeño que cae en `k = 0` no reciben rotación (limitación
inherente a `n_c` finito; se refleja en la fidelidad).

In [ ]:
def choose_t(F, n_c):
    lam_max = np.max(np.abs(np.linalg.eigvalsh(F)))
    return 2 * np.pi * ((2 ** (n_c - 1) - 1) / 2 ** n_c) / lam_max

def clock_to_k(c, n_c):
    return c if c < 2 ** (n_c - 1) else c - 2 ** n_c

def _controlled_power(F, t, j, mode, trotter_order, trotter_reps):
    # U^(2^j) = exp(i F t 2^j), controlada por un qubit
    if mode == "exact":
        return UnitaryGate(sla.expm(1j * t * (2 ** j) * F), label=f"U^{2**j}").control(1)
    if mode == "trotter":
        op = SparsePauliOp.from_operator(F).simplify()
        op = SparsePauliOp(op.paulis, np.real(op.coeffs))
        synth = SuzukiTrotter(order=trotter_order, reps=trotter_reps * 2 ** j)
        # PauliEvolutionGate implementa exp(-i H time) -> tiempo negativo para exp(+i F t)
        evo = PauliEvolutionGate(op, time=-t * (2 ** j), synthesis=synth)
        # Se decompone UNA vez para fijar la síntesis de Trotter en una compuerta con definición.
        # Si no, Statevector evalúa la PauliEvolutionGate controlada con su matriz exacta y la
        # simulación sería idéntica a `exact` (solo los recursos reflejarían Trotter).
        sub = QuantumCircuit(op.num_qubits)
        sub.append(evo, range(op.num_qubits))
        return sub.decompose().to_gate(label=f"Trotter U^{2**j}").control(1)
    raise ValueError(mode)

def build_hhl_circuit(F, b, n_c, mode="exact", t=None, trotter_order=2, trotter_reps=1):
    dim = F.shape[0]
    n_s = int(np.log2(dim))
    assert 2 ** n_s == dim, "F debe ser de lado potencia de 2"
    if t is None:
        t = choose_t(F, n_c)
    n_sys_clk = n_s + n_c
    sys_q = list(range(n_s))
    clk_q = list(range(n_s, n_s + n_c))

    qpe = QuantumCircuit(n_sys_clk, name="QPE")
    qpe.h(clk_q)
    for j in range(n_c):
        cg = _controlled_power(F, t, j, mode, trotter_order, trotter_reps)
        qpe.append(cg, [clk_q[j]] + sys_q)
    qpe.append(QFTGate(n_c).inverse(), clk_q)

    qr_s, qr_c, qr_a = QuantumRegister(n_s, "s"), QuantumRegister(n_c, "c"), QuantumRegister(1, "a")
    qc = QuantumCircuit(qr_s, qr_c, qr_a)
    qc.append(StatePreparation(b / np.linalg.norm(b)), sys_q)
    qc.compose(qpe, qubits=range(n_sys_clk), inplace=True)
    for c in range(1, 2 ** n_c):
        k = clock_to_k(c, n_c)
        theta = 2 * np.arcsin(1.0 / k)              # C / lambda_k = 1/k
        qc.append(RYGate(theta).control(n_c, ctrl_state=c), clk_q + [n_sys_clk])
    qc.compose(qpe.inverse(), qubits=range(n_sys_clk), inplace=True)
    meta = dict(t=t, n_s=n_s, n_c=n_c, n_total=n_sys_clk + 1)
    return qc, meta

def hhl_solve(F, b, n_c, mode="exact", t=None, trotter_order=2, trotter_reps=1):
    qc, meta = build_hhl_circuit(F, b, n_c, mode, t, trotter_order, trotter_reps)
    sv = Statevector(qc).data
    off = 2 ** (meta["n_s"] + n_c)                  # ancilla=1, reloj=0
    v = sv[off : off + 2 ** meta["n_s"]]
    return dict(v=v, success_prob=float(np.vdot(v, v).real), qc=qc, **meta)

def align_phase(v, x_ref):
    # La fase global de HHL no es observable; se alinea con la solución clásica solo para
    # comparar componentes (imag_residual) y el signo global al clasificar.
    xv = v / np.linalg.norm(v)
    ov = np.vdot(x_ref, xv)
    return xv * np.conj(ov / abs(ov))

## 5. Recursos (transpilación) y ruido opcional

In [ ]:
BASIS = ["sx", "rz", "cx"]

def count_resources(qc, seed):
    tqc = transpile(qc, basis_gates=BASIS, optimization_level=1, seed_transpiler=seed)
    ops = dict(tqc.count_ops())
    return tqc, dict(depth_pre=qc.depth(), depth_post=tqc.depth(), size_post=tqc.size(),
                     cx_count=int(ops.get("cx", 0)), gate_counts=json.dumps(ops, sort_keys=True))

def noisy_postselected(tqc, n_s, n_c, x_ref, p1, p2, seed):
    # Aer, matriz de densidad, depolarizante: 1q en 'sx' (rz es virtual), 2q en 'cx'
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, depolarizing_error
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["sx"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    sim = AerSimulator(method="density_matrix", noise_model=nm, seed_simulator=seed)
    c = tqc.copy()
    c.save_density_matrix()
    rho = np.asarray(sim.run(c).result().data(0)["density_matrix"])
    off = 2 ** (n_s + n_c)
    blk = rho[off : off + 2 ** n_s, off : off + 2 ** n_s]
    p = float(np.real(np.trace(blk)))
    xr = x_ref / np.linalg.norm(x_ref)
    return float(np.real(np.vdot(xr, blk @ xr)) / p), p

## 6. Una corrida (todas las métricas)

In [ ]:
def run_one(N, n_c, gamma, mode, seed, cfg):
    t0 = time.time()
    set_seeds(seed)
    kernel = cfg["kernel"]
    Xtr, ytr, Xte, yte, tr_idx = load_subsample(N, seed, cfg["n_test"])
    F, rhs = build_lssvm(Xtr, ytr, gamma, kernel)
    lam = np.linalg.eigvalsh(F)
    x_c = solve_classical(F, rhs)
    x_cn = x_c / np.linalg.norm(x_c)

    out = hhl_solve(F, rhs, n_c, mode, trotter_order=cfg["trotter_order"], trotter_reps=cfg["trotter_reps"])
    xv = align_phase(out["v"], x_cn)
    fidelity = float(abs(np.vdot(x_cn, xv)) ** 2)

    pred_h = predict(xv, Xtr, Xte, kernel)
    pred_c = predict(x_cn, Xtr, Xte, kernel)
    tqc, res = count_resources(out["qc"], cfg["transpile_seed"])

    row = dict(
        N=N, dim_F=F.shape[0], n_system=out["n_s"], n_clock=n_c, n_total_qubits=out["n_total"],
        mode=mode, gamma=gamma, kernel=kernel, seed=seed,
        kappa=float(np.max(np.abs(lam)) / np.min(np.abs(lam))),
        eigenvalues=json.dumps(lam.tolist()), t=out["t"], train_idx=json.dumps(tr_idx.tolist()),
        success_prob=out["success_prob"], fidelity=fidelity,
        imag_residual=float(np.max(np.abs(xv.imag))),
        acc_hhl=float(np.mean(pred_h == yte)), acc_classical=float(np.mean(pred_c == yte)),
        agreement=float(np.mean(pred_h == pred_c)),
        fidelity_noisy=None, success_prob_noisy=None, **res,
    )
    if cfg["noise_enabled"] and N == 3:
        nz = cfg["noise"]
        row["fidelity_noisy"], row["success_prob_noisy"] = noisy_postselected(
            tqc, out["n_s"], n_c, x_cn, nz["p1"], nz["p2"], seed)
    row["wall_time_s"] = time.time() - t0
    return row

row = run_one(CONFIG["N"][0], CONFIG["n_c"][0], CONFIG["gamma"][0], CONFIG["mode"][0], CONFIG["seeds"][0], CONFIG)
pd.Series(row).to_frame("valor")

## 7. Pruebas mínimas (`assert`)

1. La solución clásica coincide con `np.linalg.solve` y clasifica bien el entrenamiento en un caso separable trivial.
2. HHL sobre una matriz diagonal 4×4 con valores propios exactamente representables (uno negativo): `fidelity > 0.999`.
3. `N = 3`, `n_c = 6`, `mode = exact`: fidelidad alta (si < 0.9, revisar `t` y el signo).
4. El modo `trotter` realmente simula la síntesis de Trotter (difiere de `exact`, el error baja con más pasos) y la transpilación conserva el estado.

In [ ]:
# Prueba 1: LS-SVM clásica
Xs = np.array([[-2.0, 0.0], [-1.0, 0.5], [1.0, -0.5], [2.0, 0.0]])
ys = np.array([-1.0, -1.0, 1.0, 1.0])
Fs, rs = build_lssvm(Xs, ys, gamma=10.0)
assert Fs.shape == (8, 8), Fs.shape                               # N=4 -> 5, relleno a 8
assert np.allclose(Fs, Fs.T)
assert np.allclose(Fs[5:, 5:], np.eye(3)) and np.allclose(Fs[:5, 5:], 0) and np.allclose(rs[5:], 0)
xs = solve_classical(Fs, rs)
assert np.allclose(xs, np.linalg.solve(Fs, rs))
assert np.allclose(Fs @ xs, rs)
assert np.array_equal(predict(xs, Xs, Xs), ys), "no clasifica el entrenamiento separable"
assert np.sum(np.linalg.eigvalsh(Fs) < 0) == 1, "F debe tener exactamente un valor propio negativo"
print("Prueba 1 OK")

# Prueba 2: HHL en diagonal 4x4 con valores propios exactos (n_c=4, t=2π/16 -> k = λ)
Fd = np.diag([1.0, 2.0, 3.0, -2.0])
bd = np.ones(4)
od = hhl_solve(Fd, bd, n_c=4, mode="exact", t=2 * np.pi / 16)
xd = od["v"] / np.linalg.norm(od["v"])
xc = np.linalg.solve(Fd, bd); xc /= np.linalg.norm(xc)
fid = abs(np.vdot(xc, xd)) ** 2
print("Prueba 2: fidelity =", fid, "| success_prob =", od["success_prob"])
assert fid > 0.999

# Prueba 3: N=3, n_c=6, exact. gamma no está fijado por la especificación; se usa gamma=10
# (kappa ~ 40). Con kappa mayor (gamma=1: ~55; gamma=0.1: ~280) los valores propios pequeños
# caen por debajo de la resolución del reloj (k < 1) y la fidelidad baja: es una limitación de
# n_c finito, no un error de signo o de t (la prueba 2 y la convergencia con n_c lo descartan).
r3 = run_one(3, 6, 10, "exact", 0, {**CONFIG, "noise_enabled": False})
print("Prueba 3: fidelity =", r3["fidelity"], "| kappa =", r3["kappa"], "| cx =", r3["cx_count"])
assert r3["fidelity"] > 0.9, "fidelidad baja: revisar t y signo"

# Prueba 4: el modo trotter se simula de verdad (no equivale a exact) y es consistente con la transpilación
Xt, yt, _, _, _ = load_subsample(3, 0, CONFIG["n_test"])
Ft, rt = build_lssvm(Xt, yt, 10)
ve = hhl_solve(Ft, rt, 3, "exact")["v"]
d1 = np.linalg.norm(hhl_solve(Ft, rt, 3, "trotter", trotter_reps=1)["v"] - ve)
d16 = np.linalg.norm(hhl_solve(Ft, rt, 3, "trotter", trotter_reps=16)["v"] - ve)
print(f"Prueba 4: ||v_trotter - v_exact||: reps=1 -> {d1:.3e}, reps=16 -> {d16:.3e}")
assert d1 > 1e-6, "trotter da lo mismo que exact: no se está simulando la síntesis de Trotter"
assert d16 < d1, "el error de Trotter debe bajar al aumentar los pasos"
qct = hhl_solve(Ft, rt, 3, "trotter")["qc"]
tq_, _ = count_resources(qct, 0)
assert abs(np.vdot(Statevector(qct).data, Statevector(tq_).data)) ** 2 > 1 - 1e-9, "transpilación cambia el estado"
print("Prueba 4 OK")

# Informativo (sin assert): dependencia de la fidelidad con gamma y n_c, semilla 0, N=3
Xi, yi, _, _, _ = load_subsample(3, 0, CONFIG["n_test"])
for g_ in (0.1, 1, 10):
    Fi, ri = build_lssvm(Xi, yi, g_)
    xi = solve_classical(Fi, ri); xi /= np.linalg.norm(xi)
    kap = np.linalg.cond(Fi)
    fids = {nc: round(float(abs(np.vdot(xi, align_phase(hhl_solve(Fi, ri, nc)["v"], xi))) ** 2), 4) for nc in (4, 5, 6, 7)}
    print(f"gamma={g_:<4} kappa={kap:6.1f}  fidelity por n_c: {fids}")

## 8. Barrido (con checkpoint por corrida)

In [ ]:
out_dir = setup_output(OUTPUT_DIR, RESUME)
log = get_logger(out_dir)
csv_path = out_dir / "runs.csv"

with open(out_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(dict(config=CONFIG, versions=package_versions(), python=platform.python_version(),
                   date=datetime.now().isoformat(timespec="seconds")), f, indent=2)

done = load_done(csv_path) if RESUME else set()
grid = list(itertools.product(CONFIG["N"], CONFIG["n_c"], CONFIG["gamma"], CONFIG["mode"], CONFIG["seeds"]))
log.info("Salida: %s | %d corridas (%d ya hechas)", out_dir, len(grid), len(done))

for N, n_c, gamma, mode, seed in tqdm(grid):
    key = run_key(N, n_c, gamma, mode, seed, CONFIG["kernel"])
    if key in done:
        continue
    try:
        row = run_one(N, n_c, gamma, mode, seed, CONFIG)
    except Exception:
        log.exception("Falló %s", key)
        continue
    append_row(csv_path, row)
    log.info("%s fidelity=%.4f p_succ=%.4g cx=%d t=%.1fs", key, row["fidelity"], row["success_prob"],
             row["cx_count"], row["wall_time_s"])
log.info("Terminado")

## 9. Vista previa de los resultados crudos

In [ ]:
df = pd.read_csv(csv_path)
print(len(df), "filas en", csv_path)
df[["N", "n_clock", "gamma", "mode", "seed", "kappa", "depth_post", "cx_count",
    "success_prob", "fidelity", "acc_hhl", "acc_classical", "agreement", "wall_time_s"]].head(10)